#### 🥇 Pipeline de Modelagem Dimensional - Camada Gold (Ouro)

##### Objetivo
Este notebook implementa o processo ETL da **Camada Prata → Camada Gold (Ouro)**, em modelo Star Schema** com:

* **Criação de Tabelas Dimensão** (Cliente, Produto, Carro, Vendedor, Loja, Data)
* **Criação de Tabela Fato** (Vendas) com métricas calculadas
* **Consultas analíticas** pré-construídas para análises de negócio
* **Modelo otimizado** para BI e dashboards

##### Fluxo de Dados
```
tb_clean_data → vw_vendas → [Dimensões] → tb_fato_vendas → Consultas Analíticas
```

##### Arquitetura
* **Modelo**: Star Schema (Estrela)
* **Tabelas Dimensão**: 6 dimensões com surrogate keys (SK)
* **Tabela Fato**: Vendas com métricas calculadas (faturamento, lucro)
* **Técnica**: Modelagem Dimensional (Kimball)

##### Catalog e Schema
* **Catalog**: `vendas_pecas`
* **Schema**: `camada_ouro`

In [0]:
USE vendas_pecas.camada_ouro;

#### 0️⃣ Criação de View Fonte

##### Objetivo
Criar uma view temporária que serve como fonte única de dados para todas as tabelas dimensão e fato.

##### Estratégia
* `Fonte`: `tb_clean_data` (camada prata - dados limpos e tratados)
* `Filtro`: Remove registros com `IdNotaFiscal` nulo (chave primária)
* `Propósito`: Centralizar acesso aos dados e garantir consistência

##### Recursos Utilizados
* **CREATE OR REPLACE VIEW**: Cria view idempotente (pode ser recriada)
* **WHERE IS NOT NULL**: Garante integridade referencial

##### Output
View temporária: `vw_vendas` (base para todas as dimensões e fato)

In [0]:
CREATE OR REPLACE VIEW vw_vendas
AS
SELECT *
FROM 
  vendas_pecas.camada_prata.tb_clean_data cd
WHERE 
  cd.IdNotaFiscal IS NOT NULL

#### 📊 Tabelas Dimensão

##### Conceito
Dimensões são tabelas que contêm **atributos descritivos** do negócio. No modelo Star Schema, elas circundam a tabela fato e fornecem contexto para as métricas.

##### Padrão de Implementação
Cada dimensão segue este padrão de 3 etapas:

1. **CREATE TABLE**: Cria estrutura da dimensão com:
   * `sk_*` (Surrogate Key): Chave artificial auto-incremental (IDENTITY)
   * Atributos de negócio: Colunas descritivas

2. **CREATE TEMP VIEW**: Extrai dados distintos da fonte (`vw_vendas`)
   * Remove duplicatas (DISTINCT)
   * Prepara dados para carga

3. **MERGE**: Carga incremental
   * `MATCHED`: Atualiza registros existentes
   * `NOT MATCHED`: Insere novos registros
   * Garante idempotência (pode rodar múltiplas vezes)

##### Dimensões Criadas
* 💻 **tb_dim_produtos**: Catálogo de peças
* 👤 **tb_dim_cliente**: Base de clientes
* 🚗 **tb_dim_carro**: Modelos e marcas de veículos
* 👥 **tb_dim_vendedor**: Equipe de vendas
* 🏪 **tb_dim_loja**: Lojas e grupos
* 📅 **tb_dim_data_venda**: Calendário de vendas

In [0]:
USE vendas_pecas.camada_ouro;

#### 1️⃣ Dimensão de Produtos

##### Objetivo
Armazenar o catálogo de peças automotivas comercializadas.

##### Estrutura da Tabela
* `sk_produto` (PK): Surrogate Key auto-incremental (IDENTITY)
* `produto_peca`: Nome da peça (sem acentuação)

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

CREATE OR REPLACE TABLE tb_dim_produtos (
  sk_produto BIGINT GENERATED ALWAYS AS IDENTITY,
  produto_peca STRING  
);


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_produtos
AS
SELECT DISTINCT
  produto_peca_sem_Acento as produto_peca
FROM
  vw_vendas
;


-------------- MERGE -------------- 

MERGE INTO tb_dim_produtos dp
USING 
  vw_dim_produtos st
ON dp.produto_peca = st.produto_peca
WHEN 
  MATCHED 
THEN
  UPDATE SET dp.produto_peca = st.produto_peca
WHEN
  NOT MATCHED
THEN
  INSERT (produto_peca) VALUES (st.produto_peca)

       

#### 2️⃣ Dimensão de Clientes

##### Objetivo
Armazenar base de clientes com identificação e dados cadastrais.

##### Estrutura da Tabela
* `sk_cliente` (PK): Surrogate Key auto-incremental
* `cliente_id` (NK): Natural Key - ID original do cliente
* `cliente_nome`: Nome completo do cliente

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

CREATE OR REPLACE TABLE tb_dim_cliente (
  sk_cliente BIGINT GENERATED ALWAYS AS IDENTITY,
  cliente_id BIGINT,
  cliente_nome STRING  
);


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_cliente
AS
SELECT DISTINCT
  cliente_id,
  cliente_nome
FROM
  vw_vendas
;


-------------- MERGE -------------- 

MERGE INTO tb_dim_cliente dc
USING 
  vw_dim_cliente st
ON dc.cliente_id = st.cliente_id
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dc.cliente_id = st.cliente_id, 
    dc.cliente_nome = st.cliente_nome
WHEN
  NOT MATCHED
THEN
  INSERT (cliente_id, cliente_nome) VALUES (st.cliente_id, st.cliente_nome)  
  

#### 3️⃣ Dimensão de Carros

##### Objetivo
Catálogo de modelos e marcas de veículos para análise de compatibilidade de peças.

##### Estrutura da Tabela
* `sk_carro` (PK): Surrogate Key auto-incremental
* `marca_carro`: Marca do veículo (ex: Toyota, Ford)
* `modelo_carro`: Modelo específico (ex: Corolla, Fiesta)

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

CREATE OR REPLACE TABLE tb_dim_carro (
  sk_carro BIGINT GENERATED ALWAYS AS IDENTITY,
  marca_carro STRING,
  modelo_carro STRING
);


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_carro
AS
SELECT DISTINCT
  marca_carro,
  modelo_carro
FROM
  vw_vendas
;


-------------- MERGE -------------- 

MERGE INTO tb_dim_carro dc
USING 
  vw_dim_carro st
ON dc.marca_carro = st.marca_carro
AND dc.modelo_carro = st.modelo_carro
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dc.marca_carro = st.marca_carro, 
    dc.modelo_carro = st.modelo_carro
WHEN
  NOT MATCHED
THEN
  INSERT (marca_carro, modelo_carro) VALUES (st.marca_carro, st.modelo_carro)
      

#### 4️⃣ Dimensão de Vendedores

##### Objetivo
Base da equipe de vendas para avaliação de desempenho individual.

##### Estrutura da Tabela
* `sk_vendedor` (PK): Surrogate Key auto-incremental
* `vendedor_id` (NK): Código do vendedor
* `vendedor_nome`: Nome completo

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

CREATE OR REPLACE TABLE tb_dim_vendedor (
  sk_vendedor BIGINT GENERATED ALWAYS AS IDENTITY,
  vendedor_id STRING,
  vendedor_nome STRING
);


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_vendedor
AS
SELECT DISTINCT
  vendedor_id,
  vendedor_nome
FROM
  vw_vendas
ORDER BY vendedor_id
;


-------------- MERGE -------------- 

MERGE INTO tb_dim_vendedor dv
USING 
  vw_dim_vendedor st
ON dv.vendedor_id = st.vendedor_id
AND dv.vendedor_nome = st.vendedor_nome
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dv.vendedor_id = st.vendedor_id, 
    dv.vendedor_nome = st.vendedor_nome
WHEN
  NOT MATCHED
THEN
  INSERT (vendedor_id, vendedor_nome) VALUES (st.vendedor_id, st.vendedor_nome)
      

#### 5️⃣ Dimensão de Lojas

##### Objetivo
Hierarquia organizacional de lojas e grupos para análises regionais e comparações.

##### Estrutura da Tabela
* `sk_loja` (PK): Surrogate Key auto-incremental
* `loja_id` (NK): ID original da loja
* `loja_nome`: Nome da loja (sem acentuação)
* `grupo_loja`: Grupo ao qual a loja pertence (hierarquia)

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

CREATE OR REPLACE TABLE tb_dim_loja (
  sk_loja BIGINT GENERATED ALWAYS AS IDENTITY,
  loja_id BIGINT,
  loja_nome STRING,
  grupo_loja STRING
);


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_loja
AS
SELECT DISTINCT
  loja_id,
  loja_nome_sem_acento AS loja_nome,
  grupo_loja
FROM
  vw_vendas
ORDER BY loja_id
;


-------------- MERGE -------------- 

MERGE INTO tb_dim_loja dl
USING 
  vw_dim_loja st
ON dl.loja_id = st.loja_id
AND dl.loja_nome = st.loja_nome
AND dl.grupo_loja = st.grupo_loja
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    dl.loja_id = st.loja_id, 
    dl.loja_nome = st.loja_nome,
    dl.grupo_loja = st.grupo_loja
WHEN
  NOT MATCHED
THEN
  INSERT (loja_id, loja_nome, grupo_loja) VALUES (st.loja_id, st.loja_nome, st.grupo_loja)
      

#### 6️⃣ Dimensão de Data de Venda

##### Objetivo
Calendário analítico com atributos de tempo para análises temporais (time-series).

##### Estrutura da Tabela
* `sk_data_venda` (PK): Surrogate Key auto-incremental
* `data_venda` (NK): Data completa da venda
* `ano_venda`: Ano extraído (ex: 2024, 2025)
* `mes_venda`: Mês numérico (1-12)
* `dia_venda`: Dia do mês (1-31)

In [0]:
-------- CRIANDO TABELA DIMENSAO -----------

CREATE OR REPLACE TABLE tb_dim_data_venda (
  sk_data_venda BIGINT GENERATED ALWAYS AS IDENTITY,
  data_venda DATE,
  ano_venda INT,
  mes_venda INT,
  dia_venda INT
);


------------ TEMP VIEW ------------

CREATE OR REPLACE TEMP VIEW vw_dim_data_venda
AS
SELECT DISTINCT
  data_venda,
  ano_venda,
  mes_venda,
  DAY(data_venda) AS dia_venda 
FROM
  vw_vendas
ORDER BY data_venda
;


-------------- MERGE -------------- 

MERGE INTO tb_dim_data_venda ddv
USING 
  vw_dim_data_venda st
ON ddv.data_venda = st.data_venda
AND ddv.ano_venda = st.ano_venda
AND ddv.mes_venda = st.mes_venda
WHEN 
  MATCHED 
THEN
  UPDATE SET 
    ddv.data_venda = st.data_venda, 
    ddv.ano_venda = st.ano_venda,
    ddv.mes_venda = st.mes_venda,
    ddv.dia_venda = st.dia_venda
WHEN
  NOT MATCHED
THEN
  INSERT (data_venda, ano_venda, mes_venda, dia_venda) VALUES (st.data_venda, st.ano_venda, st.mes_venda, st.dia_venda)
      



## 🎯 Tabela Fato

### Conceito
A **Tabela Fato** é o centro do Star Schema, armazenando **métricas mensuráveis** (fatos) do negócio. Cada linha representa uma **transação de venda** com referências (Foreign Keys) para as dimensões.

### Características da Tabela Fato
* **Granularidade**: 1 linha por nota fiscal (venda)
* **Foreign Keys**: Ligações com todas as 6 dimensões (SKs)
* **Métricas**: Valores numéricos mensuráveis
* **Fatos Derivados**: Métricas calculadas (faturamento, lucro)

### Tipos de Fatos
1. **Fatos Aditivos** (podem ser somados em todas as dimensões):
   * `valor_unitario`, `quantidade`, `custo_unitario`
   * `faturamento` = valor_unitario * quantidade
   * `lucro` = faturamento - (custo_unitario * quantidade)

2. **Metadados de Auditoria**:
   * `file_path`: Rastreabilidade da origem
   * `ingest_date`: Data de carga

### Estrutura dos JOINs
```
vw_vendas (fonte)
  LEFT JOIN tb_dim_cliente (sk_cliente)
  LEFT JOIN tb_dim_data_venda (sk_data_venda)
  LEFT JOIN tb_dim_loja (sk_loja)
  LEFT JOIN tb_dim_produtos (sk_produto)
  LEFT JOIN tb_dim_vendedor (sk_vendedor)
  LEFT JOIN tb_dim_carro (sk_carro)
```

### Benefícios do Modelo
* ✅ Consultas analíticas otimizadas (menos JOINs)
* ✅ Agregações rápidas por qualquer dimensão
* ✅ Esquema intuitivo para ferramentas de BI

In [0]:
USE vendas_pecas.camada_ouro;

#### 🎯 Tabela Fato: Vendas

##### Objetivo
Criar a tabela central do Data Warehouse com todas as transações de venda enriquecidas com referências dimensionais.

##### Estrutura da Tabela
**Chave Primária:**
* `IdNotaFiscal`: Identificador único da venda

**Foreign Keys (Surrogate Keys das Dimensões):**
* `sk_data_venda` → Quando foi vendido
* `sk_cliente` → Quem comprou
* `sk_carro` → Para qual veículo
* `sk_loja` → Onde foi vendido
* `sk_produto` → O que foi vendido
* `sk_vendedor` → Quem vendeu

**Fatos (Métricas):**
* `valor_unitario`: Preço de venda unitário
* `quantidade`: Quantidade de peças
* `custo_unitario`: Custo unitário
* `faturamento`: **CALCULADO** = valor_unitario × quantidade
* `lucro`: **CALCULADO** = faturamento - (custo_unitario × quantidade)

**Metadados:**
* `file_path`: Arquivo origem
* `ingest_date`: Data de carga

##### Técnica: LEFT JOIN
Usa LEFT JOIN para preservar vendas mesmo se alguma dimensão não for encontrada (permite análise de dados incompletos).

##### Recursos Utilizados
* **CREATE OR REPLACE TABLE AS SELECT**: Recria tabela completa
* **TRY_CAST**: Conversão segura para DECIMAL(18,2)
* **LEFT JOIN**: Preserva todas as vendas da fonte
* **Cálculos In-Line**: faturamento e lucro calculados na query

##### Performance
* Ordenado por `IdNotaFiscal` para otimizar buscas
* Todos os JOINs resolvidos em tempo de carga (não em consultas)

In [0]:
CREATE OR REPLACE TABLE tb_fato_vendas AS
SELECT
  fv.IdNotaFiscal,
  ddv.sk_data_venda,
  dcl.sk_cliente,
  dc.sk_carro,
  dl.sk_loja,
  dpd.sk_produto,
  dv.sk_vendedor,
  fv.valor_unitario,
  fv.quantidade,
  fv.custo_unitario,
  try_cast((fv.valor_unitario * fv.quantidade) as decimal (18,2)) as faturamento,
  try_cast((faturamento - (fv.custo_unitario * fv.quantidade)) as decimal(18,2)) as lucro,
  file_path,
  ingest_date
FROM vw_vendas fv
LEFT JOIN tb_dim_cliente dcl ON fv.cliente_id = dcl.cliente_id
LEFT JOIN tb_dim_data_venda ddv ON fv.data_venda = ddv.data_venda
LEFT JOIN tb_dim_loja dl ON fv.loja_nome_sem_acento = dl.loja_nome
LEFT JOIN tb_dim_produtos dpd ON fv.produto_peca_sem_acento = dpd.produto_peca
LEFT JOIN tb_dim_vendedor dv ON fv.vendedor_nome = dv.vendedor_nome
LEFT JOIN tb_dim_carro dc ON fv.modelo_carro = dc.modelo_carro and fv.marca_carro = dc.marca_carro
ORDER BY IdNotaFiscal 
                                            

## 📊 Consultas Analíticas

### Objetivo
Conjunto de **consultas pré-construídas** que respondem perguntas estratégicas de negócio, aproveitando o modelo Star Schema para queries otimizadas.

### Padrão das Consultas
Todas seguem este template:
1. **SELECT**: Métricas agregadas (SUM, AVG, COUNT)
2. **FROM**: tb_fato_vendas
3. **JOIN**: Dimensões necessárias (poucos JOINs)
4. **WHERE**: Filtro de qualidade de dados
5. **GROUP BY**: Agregação por dimensões
6. **ORDER BY**: Ordenação para apresentação

### Filtro de Qualidade
Todas as consultas aplicam:
```sql
WHERE
  quantidade IS NOT NULL AND quantidade > 0
  AND valor_unitario > 0
  AND custo_unitario > 0
```
Garante que apenas vendas válidas sejam analisadas.

### Questões de Negócio
1. 📈 **Evolução Temporal**: Como estão as tendências ao longo do tempo?
2. 📅 **Sazonalidade**: Quais períodos tem melhor/pior performance?
3. 🏪 **Performance por Loja**: Qual o desempenho de lojas e grupos?
4. 🔁 **Comparação Anual**: Como 2024 vs 2025?
5. 📊 **Volume de Vendas**: Onde está concentrado o volume?

In [0]:
USE vendas_pecas.camada_ouro;

### 📈 Questão 1: Evolução Temporal

#### Como evoluíram nosso faturamento e resultados ao longo do tempo?

**Análise Mês a Mês:**
* Faturamento mensal
* Lucro mensal
* Margem de lucro (%)
* Quantidade de peças vendidas

**Dimensões:** Ano + Mês

**Uso:** Identificar tendências, crescimento, sazonalidade mensal

In [0]:
SELECT
  ano_venda AS ano,
  mes_venda AS mes,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN
  tb_dim_data_venda dv ON dv.sk_data_venda = fv.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  mes, ano
ORDER BY 
  ano_venda, mes_venda

### 📅 Questão 2: Sazonalidade e Periodicidade

#### Quais períodos apresentam melhor e pior performance?

**Análises Disponíveis:**

1. **Semestral** (2024-2025):
   * Comparação 1º vs 2º semestre
   * Identifica padrões semestrais

2. **Mensal** (2024-2025):
   * Performance mês a mês
   * Identifica melhores/piores meses

3. **Quinzenal** (2024):
   * Granularidade: 1ª vs 2ª quinzena do mês
   * Identifica padrões intra-mensais

**Uso:** Planejamento de estoque, promoções, força de vendas

In [0]:
SELECT 
  ano_venda,
  CASE
    WHEN mes_venda <= 6 
      THEN '1º Semestre'
    WHEN mes_venda > 6 
      THEN '2º Semestre'
  END as semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt ON fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  ano_venda, semestre
ORDER BY 
  ano_venda

In [0]:
SELECT 
    ano_venda, 
    mes_venda, 
    SUM(faturamento) AS faturamento, 
    SUM(lucro) as lucro,
    SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt on fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY  
  ano_venda, mes_venda
ORDER BY 
  ano_venda, mes_venda

In [0]:
SELECT 
  ano_venda,
  mes_venda,
  CASE
    WHEN day(data_venda) <= 15 
      THEN '1º Quinzena'
    WHEN day(data_venda) > 15 
      THEN '2º Quinzena'
  END AS Semestre,
  SUM(faturamento) AS faturamento,
  SUM(lucro) AS lucro,
  SUM(quantidade) AS quantidade_pecas
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_data_venda ddt ON fv.sk_data_venda = ddt.sk_data_venda
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  ano_venda, mes_venda, semestre
ORDER BY 
  ano_venda,mes_venda

### 🏪 Questão 3: Desempenho por Localidade

#### Qual o desempenho das vendas por loja, grupo de lojas e vendedor?

**Métricas de Desempenho:**
* Faturamento total
* Lucro total
* Margem de lucro (%)
* Ticket médio (faturamento / nº vendas)
* Lucro médio unitário

**Análises Disponíveis:**

1. **Por Grupo de Loja e Loja Individual**:
   * Hierarquia: Grupo → Loja
   * Ranking de performance
   * Identifica melhores/piores unidades

2. **Por Vendedor**:
   * Performance individual
   * Rankings de vendedores
   * Base para comissões e incentivos

**Uso:** Avaliação de performance, planejamento estratégico regional

In [0]:
SELECT 
  grupo_loja,
  loja_nome, 
  SUM(faturamento) AS faturamento, 
  SUM(lucro) AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio,
  SUM(lucro) / SUM(quantidade) AS lucro_medio_unitario
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON fv.sk_loja = dl.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
   grupo_loja, loja_nome
ORDER BY 
  faturamento desc

In [0]:
SELECT
  vendedor_nome,
  SUM(faturamento) AS faturamento,
  SUM(lucro)AS lucro,
  SUM(lucro) / SUM(faturamento) * 100 AS margem_lucro,
  SUM(faturamento) / count(IdNotaFiscal) AS ticket_medio,
  SUM(lucro) / SUM(quantidade) AS lucro_medio_unitario
FROM 
  tb_fato_vendas fv
JOIN 
  tb_dim_vendedor dv ON fv.sk_vendedor = dv.sk_vendedor
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  and
  dv.sk_vendedor = fv.sk_vendedor

GROUP BY 
  vendedor_nome
ORDER BY 
  faturamento DESC
   

### 🔁 Questão 4: Comparação entre Anos

#### Como está o comparativo entre anos?

**Métricas Comparadas (2024 vs 2025):**
* Faturamento total
* Lucro total
* Custo total
* Quantidade de peças vendidas
* Número de vendas

**Técnica:**
* Usa `ROW_NUMBER()` para ranking dentro de cada ano
* Permite comparação YoY (Year-over-Year)

**Uso:** Avaliação de crescimento, identificação de tendências ano a ano

In [0]:
SELECT
    ano_venda AS ano,
    SUM(faturamento) AS faturamento, 
    SUM(lucro) AS lucro,  
    SUM(custo_unitario * quantidade) AS custo_total,
    SUM(quantidade) AS qtd_pecas_vendidas,
    COUNT(IdNotaFiscal) AS n_vendas,
    ROW_NUMBER() OVER(
      PARTITION BY ano_venda
      ORDER BY sum(faturamento) DESC
  ) AS rn_quantidade
  FROM 
    tb_fato_vendas fv
  JOIN 
    tb_dim_data_venda ddt ON ddt.sk_data_venda = fv.sk_data_venda
  JOIN
    tb_dim_produtos dp ON dp.sk_produto = fv.sk_produto
  WHERE
    fv.quantidade IS NOT NULL and
    fv.quantidade > 0 and
    fv.valor_unitario > 0 and
    fv.custo_unitario > 0
  GROUP BY  ano

### 📊 Questão 5: Concentração de Volume

#### Quais unidades e grupos concentram maior volume de vendas?

**Objetivo:**
Identificar onde está concentrado o **volume de transações** (número de vendas) versus o **volume de produtos** (quantidade vendida).

**Métricas:**
* Contagem de vendas por grupo/loja
* Quantidade de peças vendidas
* Distribuição percentual

**Uso:** Planejamento logístico, alocação de recursos, estratégia de distribuição

In [0]:
SELECT 
  grupo_loja,
  loja_nome,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.sk_loja = fv.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0  
GROUP BY 
   grupo_loja, loja_nome
ORDER BY n_vendas DESC


In [0]:
SELECT 
  grupo_loja,
  count(IdNotaFiscal) AS n_vendas,
  SUM(quantidade) AS qtd_vendida
FROM
  tb_fato_vendas fv
JOIN 
  tb_dim_loja dl ON dl.sk_loja = fv.sk_loja
WHERE
  fv.quantidade IS NOT NULL and
  fv.quantidade > 0 and
  fv.valor_unitario > 0 and
  fv.custo_unitario > 0
GROUP BY 
  grupo_loja
ORDER BY n_vendas DESC

## ✅ Resumo: Camada Gold (Ouro)

### Arquitetura Implementada

**Modelo Dimensional: Star Schema**
```
                    tb_dim_data_venda
                           |
    tb_dim_cliente ----    |    ---- tb_dim_carro
                       \   |   /
                        \  |  /
                    tb_fato_vendas
                        /  |  \
                       /   |   \
    tb_dim_produto ----    |    ---- tb_dim_loja
                           |
                    tb_dim_vendedor
```

### Componentes Criados

#### 📊 6 Tabelas Dimensão
1. **tb_dim_produtos**: Catálogo de peças
2. **tb_dim_cliente**: Base de clientes
3. **tb_dim_carro**: Modelos e marcas
4. **tb_dim_vendedor**: Equipe de vendas
5. **tb_dim_loja**: Lojas e grupos (hierarquia)
6. **tb_dim_data_venda**: Calendário analítico

#### 🎯 1 Tabela Fato
* **tb_fato_vendas**: Transações com métricas calculadas
  * Faturamento (valor_unitario × quantidade)
  * Lucro (faturamento - custos)
  * Referências para todas as dimensões

#### 📊 10+ Consultas Analíticas
Queries pré-construídas para:
* Evolução temporal (mensal, semestral, quinzenal)
* Performance por loja e grupo
* Performance por vendedor
* Comparação entre anos
* Análise de volume

### Benefícios do Modelo

✅ **Performance**: Consultas otimizadas com poucos JOINs
✅ **Simplicidade**: Esquema intuitivo para usuários de negócio
✅ **Escalabilidade**: Fácil adição de novas dimensões/métricas
✅ **Compatibilidade**: Ideal para ferramentas de BI (Power BI, Tableau, Looker)
✅ **Manutenção**: Modelo padronizado e document
✅ **Idempotência**: MERGE garante que o processo pode rodar múltiplas vezes

---

**Pipeline Completo**: Bronze → Silver → **Gold** ✅